<a href="https://colab.research.google.com/github/Derrickdc02/DIS-Project-Lensed-Galaxy/blob/main/notebooks/PQMassPriorCheck_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Prior Check via PQMass (Colab)

Colab-adapted version of `notebooks/PQMassPriorCheck.ipynb`: validate the trained
256x256 score-based prior by testing whether the *unconditional* draws
`x ~ p(x)` saved on Google Drive are statistically indistinguishable from the
real PROBES galaxies, using [PQMass](https://github.com/Ciela-Institute/PQM)
(a Voronoi-tessellation two-sample chi^2 test).

Self-contained: no repo clone and no model/GPU needed -- the sampling is already
done, so a **CPU runtime is fine**. Requirements on Drive:

1. the prior draws (a single `.pt` tensor `(N, 1, 256, 256)` or a `chunks/` dir
   of `chunk_*.pt` files from `src/sample_prior.py`),
2. the preprocessed real galaxies `gals_gband_norm/*.npy` (each `(1, H, W)` or
   `(H, W)`, values in [-1, 1]). This dir is untracked in git, so copy it to
   Drive from the HPC or your local machine if it is not there yet.

## 1. Setup

In [ ]:
!pip install -q pqm

In [ ]:
from pathlib import Path
import glob

import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from scipy.stats import chi2 as chi2_dist
from pqm import pqm_chi2, pqm_pvalue

from google.colab import drive
drive.mount('/content/drive')

# ---- Adjust these two paths to where your run lives on Drive ----
DRIVE = Path('/content/drive/MyDrive/DIS-Project-Lensed-Galaxy')
PRIOR_PATH = DRIVE / 'probes_final' / 'prior_check'   # dir with chunks/ or prior_samples.pt, or a direct .pt file
DATA_DIR   = DRIVE / 'data' / 'gals_gband_norm'       # real preprocessed PROBES .npy files

IMAGE_SIZE = 256
SEED = 21

## 2. Load prior samples + real PROBES data

Accepts whatever the sampling run left behind: a directory of `chunk_*.pt`
files, a directory containing `prior_samples.pt`, or a direct `.pt` file.
Match the two set sizes so neither dominates the tessellation, then flatten
to `(N, 65536)` for PQMass.

In [ ]:
def load_prior(path):
    """Load prior draws from a .pt file or a directory (chunks/ or prior_samples.pt)."""
    path = Path(path)
    if path.is_file():
        return torch.load(path, map_location='cpu')
    chunks = sorted(glob.glob(str(path / 'chunks' / 'chunk_*.pt'))) \
             or sorted(glob.glob(str(path / 'chunk_*.pt')))
    if chunks:
        print(f'Concatenating {len(chunks)} chunk files')
        return torch.cat([torch.load(f, map_location='cpu') for f in chunks], dim=0)
    merged = path / 'prior_samples.pt'
    if merged.exists():
        return torch.load(merged, map_location='cpu')
    raise FileNotFoundError(f'No prior samples found under {path}')


prior = load_prior(PRIOR_PATH)                # (N, 1, H, W)
prior = prior.clamp(-1.0, 1.0)   # match the data domain: PROBES is hard-clipped to
                                 # [-1,1] in preprocessing, but the continuous sampler
                                 # overshoots (~-1.04/1.07); that boundary mismatch alone
                                 # roughly doubles the pixel-space chi^2.
print(f'Loaded {prior.shape[0]} prior samples, shape {tuple(prior.shape)}, '
      f'range=[{prior.min():.3f}, {prior.max():.3f}]')

# Real PROBES galaxies (inline version of train_prior.load_probes)
files = sorted(glob.glob(str(DATA_DIR / '*.npy')))
assert files, f'No .npy files found in {DATA_DIR} -- copy gals_gband_norm to Drive first'
real = np.stack([np.load(f) for f in files]).astype(np.float32)
if real.ndim == 4:
    real = real[:, 0]                          # (M, H, W)
if real.shape[1] != IMAGE_SIZE:
    real = F.interpolate(torch.from_numpy(real).unsqueeze(1),
                         size=(IMAGE_SIZE, IMAGE_SIZE),
                         mode='bilinear', align_corners=False).squeeze(1).numpy()
    print(f'Resized real images to {IMAGE_SIZE}x{IMAGE_SIZE}')
real = torch.from_numpy(real).unsqueeze(1)     # (M, 1, H, W)
print(f'Loaded {real.shape[0]} real galaxies')

n = min(prior.shape[0], real.shape[0])
g = torch.Generator().manual_seed(SEED)
real = real[torch.randperm(real.shape[0], generator=g)[:n]]
prior = prior[:n]
print(f'Using {n} samples per set')

x_real = real.reshape(n, -1).numpy().astype(np.float32)
x_prior = prior.reshape(n, -1).numpy().astype(np.float32)

## 3. Visual sanity check

Eyeball real vs prior samples (log-flux, repo convention) and compare a couple
of per-image summary statistics before trusting the formal test.

In [ ]:
FLUX_A = 5.5  # PROBES normalization constant (matches data/preprocess.py)

def to_display_flux(img, floor=1e-3):
    """[-1,1] intensity -> PROBES flux [0, FLUX_A], floored >0 for LogNorm."""
    return np.clip(FLUX_A * (np.asarray(img) + 1.0) / 2.0, floor, None)

sel = np.random.default_rng(0).choice(n, size=6, replace=False)
fig, axes = plt.subplots(2, 6, figsize=(14, 5))
flux_kw = dict(cmap='magma', norm=LogNorm(vmin=1e-2, vmax=5.5), origin='lower')
for col, j in enumerate(sel):
    axes[0, col].imshow(to_display_flux(real[j, 0]), **flux_kw)
    axes[1, col].imshow(to_display_flux(prior[j, 0]), **flux_kw)
    for i in range(2):
        axes[i, col].axis('off')
axes[0, 0].set_title('real PROBES', loc='left')
axes[1, 0].set_title('prior samples', loc='left')
plt.tight_layout(); plt.show()

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for k, (name, fn) in enumerate([('total flux', lambda a: a.sum(1)),
                                ('peak intensity', lambda a: a.max(1))]):
    ax[k].hist(fn(x_real), bins=40, density=True, alpha=0.6, label='real')
    ax[k].hist(fn(x_prior), bins=40, density=True, alpha=0.6, label='prior')
    ax[k].set_title(name); ax[k].legend()
plt.tight_layout(); plt.show()

## 4. PQMass - pixel space (primary)

`pqm_chi2` / `pqm_pvalue` over many re-tessellations. If prior and data match:
`chi^2 / dof ~ 1` and the p-values are ~Uniform(0, 1) (mean ~ 0.5).

In [ ]:
RE_TESS = 200
# num_refs must be < the samples per set, ideally << it. The real-vs-real sanity
# below uses n//2 per set, so derive num_refs from n; with the full 1024-draw run
# this resolves to the standard 100.
NUM_REFS = min(100, max(2, n // 8))
KW = dict(num_refs=NUM_REFS, re_tessellation=RE_TESS, z_score_norm=True, gauss_frac=0.05)
dof = NUM_REFS - 1
print(f'n={n} per set -> num_refs={NUM_REFS}, dof={dof}')

chi2_vals = np.asarray(pqm_chi2(x_prior, x_real, **KW))
pvals = np.asarray(pqm_pvalue(x_prior, x_real, **KW))
print(f'pixel space:  chi2/dof = {chi2_vals.mean() / dof:.3f}  (target ~1)')
print(f'              p-value mean = {pvals.mean():.3f}  (target ~0.5)')

# Sanity: real vs real (disjoint halves) must not be flagged as different.
half = n // 2
p_rr = np.asarray(pqm_pvalue(x_real[:half], x_real[half:2 * half], **KW))
print(f'sanity (real vs real): p-value mean = {p_rr.mean():.3f}  (should be ~0.5)')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(chi2_vals, bins=30, density=True, alpha=0.6, label='PQMass')
xs = np.linspace(chi2_vals.min(), chi2_vals.max(), 200)
ax[0].plot(xs, chi2_dist.pdf(xs, dof), 'k--', label=f'chi2(dof={dof})')
ax[0].set_title('chi^2 (prior vs real)'); ax[0].legend()
ax[1].hist(pvals, bins=20, range=(0, 1), density=True, alpha=0.6, label='prior vs real')
ax[1].hist(p_rr, bins=20, range=(0, 1), density=True, alpha=0.6, label='real vs real')
ax[1].axhline(1.0, color='k', ls='--', label='uniform')
ax[1].set_title('p-values'); ax[1].legend()
plt.tight_layout(); plt.show()

## 5. PQMass - PCA cross-check

65,536 pixel dims is low-power for nearest-reference tessellation. Project both
sets onto the top components of the *combined* data (so the basis is neutral)
and rerun. Agreement with the pixel-space verdict is reassuring.

In [ ]:
N_COMPONENTS = 50
combined = np.concatenate([x_real, x_prior], axis=0)
mean = combined.mean(0, keepdims=True)
# Economy SVD of the centered stack; rows of Vt are the principal directions.
_, _, Vt = np.linalg.svd(combined - mean, full_matrices=False)
comps = Vt[:N_COMPONENTS]                       # (k, D)
z_real = (x_real - mean) @ comps.T              # (n, k)
z_prior = (x_prior - mean) @ comps.T

chi2_pca = np.asarray(pqm_chi2(z_prior, z_real, **KW))
p_pca = np.asarray(pqm_pvalue(z_prior, z_real, **KW))
print(f'PCA({N_COMPONENTS}):  chi2/dof = {chi2_pca.mean() / dof:.3f}   p-value mean = {p_pca.mean():.3f}')

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(chi2_pca, bins=30, density=True, alpha=0.6, label='PQMass (PCA)')
xs = np.linspace(chi2_pca.min(), chi2_pca.max(), 200)
ax[0].plot(xs, chi2_dist.pdf(xs, dof), 'k--', label=f'chi2(dof={dof})')
ax[0].set_title(f'chi^2 in PCA({N_COMPONENTS}) space'); ax[0].legend()
ax[1].hist(p_pca, bins=20, range=(0, 1), density=True, alpha=0.6)
ax[1].axhline(1.0, color='k', ls='--', label='uniform')
ax[1].set_title('p-values (PCA)'); ax[1].legend()
plt.tight_layout(); plt.show()

## 6. Interpretation

- **chi^2 / dof ~ 1** and **p-values ~ Uniform(0, 1)** (mean ~ 0.5): the prior is
  statistically indistinguishable from the data over the regions probed.
- **chi^2 / dof >> 1**, **p-values piled near 0**: the prior and data differ
  (mode collapse, wrong support, blurry/sharp mismatch). With this run's 2000
  Euler-Maruyama steps a soft/half-denoised residual is a prime suspect: the
  batch script used 4000 steps, and VE with sigma_max=263 is very sensitive to
  the step count. If the visual check in section 3 looks hazy, resample with
  more steps before blaming the model.
- **chi^2 near 0 / spurious p-values**: too few samples for the dimensionality,
  or duplicated/memorized samples. Increase the sample count and/or
  `re_tessellation` and re-check.

The pixel-space and PCA cross-check should tell the same story; a disagreement
usually means the pixel-space test is under-powered at the current sample size.